In [ ]:
%load_ext autoreload
%autoreload 2

from marketrank.spark import get_spark
from marketrank import ingest, checks
import duckdb

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
spark = get_spark()

In [2]:
spark.sql("ALTER TABLE local.raw.transactions ADD COLUMN promo_flag BOOLEAN")

spark.table("local.raw.transactions").select(
    "t_dat", "article_id", "promo_flag"
).show(5)

+----------+----------+----------+
|     t_dat|article_id|promo_flag|
+----------+----------+----------+
|2019-05-08|0156231001|      NULL|
|2019-05-08|0762656001|      NULL|
|2019-05-08|0689814008|      NULL|
|2019-05-08|0689814001|      NULL|
|2019-05-08|0572797001|      NULL|
+----------+----------+----------+
only showing top 5 rows



In [ ]:
duckdb.sql("SELECT * FROM '../data/raw/customers.csv' LIMIT 10").show()

┌──────────────────────────────────────────────────────────────────┬────────┬────────┬────────────────────┬────────────────────────┬───────┬──────────────────────────────────────────────────────────────────┐
│                           customer_id                            │   FN   │ Active │ club_member_status │ fashion_news_frequency │  age  │                           postal_code                            │
│                             varchar                              │ double │ double │      varchar       │        varchar         │ int64 │                             varchar                              │
├──────────────────────────────────────────────────────────────────┼────────┼────────┼────────────────────┼────────────────────────┼───────┼──────────────────────────────────────────────────────────────────┤
│ 00000dbacae5abe5e23885899a1fa44253a17956c6d1c3d25f88aa139fdfc657 │   NULL │   NULL │ ACTIVE             │ NONE                   │    49 │ 52043ee2162cf5aa7ee79974281

In [ ]:
spark = get_spark()
ingest.create_tables(spark)
spark.sql(f"DESCRIBE {ingest.CUSTOMERS_TABLE}").show()

:: loading settings :: url = jar:file:/Users/test/Developer/marketrank/.venv/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /Users/test/.ivy2/cache
The jars for the packages stored in: /Users/test/.ivy2/jars
org.apache.iceberg#iceberg-spark-runtime-3.5_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-5a46c5cd-8d5f-4057-ba75-b8fcaa0cd648;1.0
	confs: [default]
	found org.apache.iceberg#iceberg-spark-runtime-3.5_2.12;1.11.0 in central
:: resolution report :: resolve 80ms :: artifacts dl 2ms
	:: modules in use:
	org.apache.iceberg#iceberg-spark-runtime-3.5_2.12;1.11.0 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   1   |   0   |   0   |   0   ||   1   |   0   |
	---------------------------------------------------------------------
:: retrieving :: org.apache.spa

+--------------------+---------+-------+
|            col_name|data_type|comment|
+--------------------+---------+-------+
|         customer_id|   string|   NULL|
|                  FN|   double|   NULL|
|              Active|   double|   NULL|
|  club_member_status|   string|   NULL|
|fashion_news_freq...|   string|   NULL|
|                 age|      int|   NULL|
|         postal_code|   string|   NULL|
+--------------------+---------+-------+



In [13]:
ingest.create_tables(spark)
ingest.load_customers(spark)
spark.sql(f"SELECT COUNT(*) FROM {ingest.CUSTOMERS_TABLE}").show()

+--------+
|count(1)|
+--------+
| 1371980|
+--------+



In [14]:
ingest.create_tables(spark)
ingest.load_articles(spark)

26/08/14 23:20:02 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


In [15]:
spark.sql(f"SELECT COUNT(*) FROM {ingest.ARTICLES_TABLE}").show()

+--------+
|count(1)|
+--------+
|  105542|
+--------+



In [16]:
spark.sql(f"""
    SELECT article_id, product_code, colour_group_code, prod_name
    FROM {ingest.ARTICLES_TABLE} LIMIT 5
""").show()

+----------+------------+-----------------+--------------------+
|article_id|product_code|colour_group_code|           prod_name|
+----------+------------+-----------------+--------------------+
|0147339034|     0147339|               10|          6P SS BODY|
|0156227002|     0156227|               13|    Box 4p Kneehighs|
|0250099001|     0250099|               09|Mama Heavy Plain ...|
|0291333014|     0291333|               73|  2-p Keri tights SG|
|0293433047|     0293433|               07|Basic 2PACK tight...|
+----------+------------+-----------------+--------------------+



In [17]:
spark.sql(f"""
    SELECT COUNT(*) AS matched
    FROM {ingest.TRANSACTIONS_TABLE} t
    JOIN {ingest.ARTICLES_TABLE} a USING (article_id)
""").show()

+--------+
| matched|
+--------+
|31788324|
+--------+



In [ ]:
importlib.reload(ingest); importlib.reload(checks)

ingest.load_transactions(spark, "2019-06-01", "2019-06-01")
ingest.load_transactions(spark, "2019-06-01", "2019-06-01")

snaps = checks.snapshot_ids(spark, ingest.TRANSACTIONS_TABLE)
a = checks.read_snapshot(spark, ingest.TRANSACTIONS_TABLE, snaps[-2])
b = checks.read_snapshot(spark, ingest.TRANSACTIONS_TABLE, snaps[-1])

checks.assert_identical(a, b)                     # should PASS
checks.assert_identical(a, b, ignore_cols=())     # should FAIL